# \[CDISC\] CDISC ADaM Clinical Trial Analysis

SEOYEON CHOI  
2026-03-03

# ADaM Analysis Data Model

> The **ADaM (Analysis Data Model)** dataset is a CDISC standard for
> organizing clinical trial data to support statistical analysis,
> regulatory reporting, and traceability. Derived from **SDTM (Study
> Data Tabulation Model)** data, ADaM datasets are designed to be
> **“analysis-ready”** for generating tables, listings, and figures.
> They are required by **regulatory agencies like the FDA and PMDA**.

# Reference

-   [Phase II Alzheimer’s clinical trial
    Data](https://github.com/cdisc-org/sdtm-adam-pilot-project/tree/master)
    -   [adamdata
        guide](https://github.com/cdisc-org/sdtm-adam-pilot-project/blob/master/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/analysis/adam/datasets/dataguide.pdf)
-   [Explanation](https://www.lexjansen.com/pharmasug/2012/DS/PharmaSUG-2012-DS18.pdf)

**Methodology**

> This was a prospective, randomized, multi-center, double-blind,
> placebo-controlled, parallel-group study. Subjects were randomized
> equally to placebo, xanomeline low dose, or xanomeline

**Study Overview**

-   Study design:
    -   Phase II randomized clinical trial
    -   3 treatment arms (Placebo / Low dose / High dose)
    -   Duration: 26 weeks
    -   Population: Mild–moderate Alzheimer’s disease patients
-   Primary objectives
    -   Evaluate efficacy of active drug vs placebo
    -   Assess safety and tolerability
-   Endpoints
    -   Primary efficacy:
        -   ADAS-Cog score
    -   Secondary:
        -   CIBIC+
        -   NPI
    -   Safety
        -   Adverse events
        -   Laboratory tests
        -   Vital signs

# Import

In [458]:
import pandas as pd
import numpy as np
import pyreadstat
import os
from pathlib import Path

from scipy import stats

import statsmodels.api as sm
import statsmodels.formula.api as smf

import patsy

# Data

In [3]:
data_folder = Path("../../../../delete/adam/")

files = [
    "adae.xpt",
    "adlbc.xpt",
    "adlbh.xpt",
    "adlbhy.xpt",
    "adqsadas.xpt",
    "adqscibc.xpt",
    "adqsnpix.xpt",
    "adsl.xpt",
    "adtte.xpt",
    "advs.xpt"
]

In [4]:
data = {}
metadata = {}

In [5]:
for f in files:
    full_path = data_folder / f
    df, meta = pyreadstat.read_xport(full_path)
    
    key = f.replace(".xpt","")
    
    data[key] = df
    metadata[key] = meta
    
    print(f"{f} loaded:", df.shape)

adae.xpt loaded: (1191, 55)
adlbc.xpt loaded: (74264, 46)
adlbh.xpt loaded: (49932, 46)
adlbhy.xpt loaded: (9954, 43)
adqsadas.xpt loaded: (12463, 40)
adqscibc.xpt loaded: (730, 36)
adqsnpix.xpt loaded: (31140, 41)
adsl.xpt loaded: (254, 48)
adtte.xpt loaded: (254, 26)
advs.xpt loaded: (32139, 34)

------------------------------------------------------------------------

-   How could get the information of dataset?

In [374]:
metadata['adsl'].column_names_to_labels

------------------------------------------------------------------------

# ADaM Data Explanation

-   ADAS-Cog (Alzheimer’s Disease Assessment Scale – Cognitive Subscale)
    -   ADAS-Cog score

In [398]:
metadata['adqsadas'].column_names_to_labels

-   Clinician’s Interview-Based Impression of Change

In [399]:
metadata['adqscibc'].column_names_to_labels

# Summary

## Analysis Populations

-   Population Summary

In [342]:
df = data['adsl']
rows = ['SAFFL','ITTFL','EFFFL']

result = []

for flag in rows:

    ct = pd.crosstab(df[flag], df['ARM']).reindex(['Y','N'], fill_value=0)
    
    pct = ct.div(ct.sum(axis=0), axis=1) * 100
    
    formatted = ct.astype(str) + " (" + pct.round(1).astype(str) + "%)"
    
    total_ct = df[flag].value_counts().reindex(['Y','N'], fill_value=0)
    total_pct = total_ct / len(df) * 100
    
    formatted['Total'] = (
        total_ct.astype(str) + " (" +
        total_pct.round(0).astype(int).astype(str) + "%)"
    )
    
    formatted.index = [f"{flag}={i}" for i in formatted.index]
    
    result.append(formatted)

table = pd.concat(result)

table

## Demographics and Baseline Characteristics

-   Describe comparability of treatment groups

In [365]:
def _fmt_p(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def _mean_sd(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) == 0:
        return ""
    return f"{x.mean():.1f} ({x.std(ddof=1):.1f})"


def _n_pct(n, denom):
    if denom == 0:
        return f"{n} (0.0%)"
    return f"{n} ({100*n/denom:.1f}%)"


def _shapiro_p(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    n = len(x)
    if n < 3:
        return np.nan
    if n > 5000:
        x = x.sample(5000, random_state=1)
    try:
        return stats.shapiro(x).pvalue
    except Exception:
        return np.nan


def _cont_pvalue(df, var, arm_col, normality=True, alpha=0.05):
    groups = []
    for _, sub in df.groupby(arm_col, dropna=False):
        x = pd.to_numeric(sub[var], errors="coerce").dropna().values
        groups.append(x)

    nonempty = [g for g in groups if len(g) > 0]
    if len(nonempty) < 2:
        return (np.nan, "NA")

    if normality:
        ps = []
        for _, sub in df.groupby(arm_col, dropna=False):
            ps.append(_shapiro_p(sub[var]))

        # any NaN (too small n) or any p<=alpha -> Kruskal (conservative)
        if any(pd.isna(p) for p in ps) or any(p <= alpha for p in ps):
            try:
                return (stats.kruskal(*nonempty).pvalue, "Kruskal–Wallis")
            except Exception:
                return (np.nan, "Kruskal–Wallis")
        else:
            try:
                return (stats.f_oneway(*nonempty).pvalue, "ANOVA")
            except Exception:
                return (np.nan, "ANOVA")
    else:
        try:
            return (stats.f_oneway(*nonempty).pvalue, "ANOVA")
        except Exception:
            return (np.nan, "ANOVA")


def _cat_pvalue(df, var, arm_col):
    tab = pd.crosstab(df[var], df[arm_col], dropna=False)

    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return (np.nan, "NA")

    try:
        chi2, p, dof, expected = stats.chi2_contingency(tab.values, correction=False)
    except Exception:
        return (np.nan, "Chi-square")

    # 2x2 and any expected <5 -> Fisher
    if tab.shape == (2, 2) and (expected < 5).any():
        try:
            _, p_f = stats.fisher_exact(tab.values)
            return (p_f, "Fisher’s exact")
        except Exception:
            return (p, "Chi-square")

    return (p, "Chi-square")


def make_table1_adsl(
    df,
    arm_col="ARM",
    continuous=None,
    categorical=None,
    labels=None,
    normality_for_continuous=True,
    include_missing_row=True,
    pct_decimals=1,
    cont_decimals=1,
):
    continuous = continuous or []
    categorical = categorical or []
    labels = labels or {}

    # ARM levels in appearance order (keep NaN last)
    arm_levels = df[arm_col].dropna().unique().tolist()
    if df[arm_col].isna().any():
        arm_levels = arm_levels + [np.nan]

    total_n = len(df)

    # Footnote letters by test
    test_to_letter = {}
    letters = list("123456789")

    def _letter_for(test_name):
        if test_name in ("", "NA", None):
            return ""
        if test_name not in test_to_letter:
            test_to_letter[test_name] = letters[len(test_to_letter)]
        return test_to_letter[test_name]

    def _get_arm_subset(arm):
        if pd.isna(arm):
            return df[df[arm_col].isna()]
        return df[df[arm_col] == arm]

    def _n_pct_fmt(n, denom):
        if denom == 0:
            return f"{n} (0.{ '0'*pct_decimals }%)" if pct_decimals > 0 else f"{n} (0%)"
        fmt = f"{{:.{pct_decimals}f}}"
        return f"{n} ({fmt.format(100*n/denom)}%)"

    def _mean_sd_fmt(x):
        x = pd.to_numeric(x, errors="coerce").dropna()
        if len(x) == 0:
            return ""
        fmt = f"{{:.{cont_decimals}f}}"
        return f"{fmt.format(x.mean())} ({fmt.format(x.std(ddof=1))})"

    rows = []
    index = []

    def _add_row(row_label, values_by_arm, pval=None, test_name=None):
        row = {}
        for arm in arm_levels:
            row[arm] = values_by_arm.get(arm, "")
        row["Total"] = values_by_arm.get("Total", "")
        if pval is None or test_name in ("", "NA", None):
            row["p-value"] = ""
        else:
            lt = _letter_for(test_name)
            row["p-value"] = f"{_fmt_p(pval)}[{lt}]"
        rows.append(row)
        index.append(row_label)

    # --------- CATEGORICAL ---------
    for var in categorical:
        var_label = labels.get(var, var)

        p, test = _cat_pvalue(df, var, arm_col)

        # header row (blank cells + p-value)
        empty_vals = {arm: "" for arm in arm_levels}
        empty_vals["Total"] = ""
        _add_row(var_label, empty_vals, p, test)

        levels = pd.Series(df[var].dropna().unique()).tolist()
        for lvl in levels:
            vals = {}
            for arm in arm_levels:
                sub = _get_arm_subset(arm)
                n = (sub[var] == lvl).sum()
                denom = len(sub)
                vals[arm] = _n_pct_fmt(n, denom)
            vals["Total"] = _n_pct_fmt((df[var] == lvl).sum(), total_n)
            _add_row(f"  {lvl}", vals)

        if include_missing_row:
            vals = {}
            for arm in arm_levels:
                sub = _get_arm_subset(arm)
                n_miss = sub[var].isna().sum()
                denom = len(sub)
                vals[arm] = _n_pct_fmt(n_miss, denom)
            vals["Total"] = _n_pct_fmt(df[var].isna().sum(), total_n)
            # _add_row("  Missing", vals)

    # --------- CONTINUOUS ---------
    for var in continuous:
        var_label = labels.get(var, var)

        vals = {}
        for arm in arm_levels:
            sub = _get_arm_subset(arm)
            vals[arm] = _mean_sd_fmt(sub[var])
        vals["Total"] = _mean_sd_fmt(df[var])

        p, test = _cont_pvalue(df, var, arm_col, normality=normality_for_continuous)
        _add_row(var_label, vals, p, test)

        if include_missing_row:
            vals_m = {}
            for arm in arm_levels:
                sub = _get_arm_subset(arm)
                n_miss = pd.to_numeric(sub[var], errors="coerce").isna().sum()
                denom = len(sub)
                vals_m[arm] = _n_pct_fmt(n_miss, denom)
            vals_m["Total"] = _n_pct_fmt(pd.to_numeric(df[var], errors="coerce").isna().sum(), total_n)
            # _add_row("  Missing", vals_m)

    out = pd.DataFrame(rows, index=index)

    # Rename NaN ARM column if exists
    col_rename = {}
    for arm in arm_levels:
        if pd.isna(arm):
            col_rename[arm] = "Missing ARM"
        else:
            col_rename[arm] = str(arm)
    out = out.rename(columns=col_rename)

    # Footnotes
    letter_to_test = {v: k for k, v in test_to_letter.items()}
    footnotes = [f"{lt} {letter_to_test[lt]}" for lt in sorted(letter_to_test.keys())]

    return out, footnotes


# =========================
# Usage
# =========================
df = data["adsl"]

labels = {
    "SEX": "Sex",
    "AGE": "Age",
    "RACE": "Race",
    "ETHNIC": "Ethnicity",
    "BMIBL": "Baseline BMI (kg/m^2)",
    "HEIGHTBL": "Baseline Height (cm)",
    "WEIGHTBL": "Baseline Weight (kg)",
    "EDUCLVL": "Years of Education",
}

categorical = ["SEX", "RACE", "ETHNIC"]
continuous = ["AGE", "BMIBL", "HEIGHTBL", "WEIGHTBL", "EDUCLVL"]

table1, footnotes = make_table1_adsl(
    df,
    arm_col="ARM",
    continuous=continuous,
    categorical=categorical,
    labels=labels,
    normality_for_continuous=False,
    include_missing_row=True,
    pct_decimals=0, 
    cont_decimals=0,
)

display(table1)

print("Footnotes:")
for f in footnotes:
    print(f)

Footnotes:
1 Chi-square
2 ANOVA

## Treatment Exposure

-   exposure duration

In [392]:
exposure = data['adsl'].groupby('ARM')['TRTDUR'].agg(
    N='count',
    Mean='mean',
    SD='std',
    Median='median',
    Min='min',
    Max='max'
)

exposure['Mean (SD)'] = exposure['Mean'].round(1).astype(str) + \
                        " (" + exposure['SD'].round(1).astype(str) + ")"

exposure['Median'] = exposure['Median'].round(1)

exposure['Min, Max'] = exposure['Min'].astype(int).astype(str) + \
                       ", " + exposure['Max'].astype(int).astype(str)

exposure = exposure[['N','Mean (SD)','Median','Min, Max']]

total = pd.DataFrame({
    'N':[df['TRTDUR'].count()],
    'Mean (SD)':[f"{df['TRTDUR'].mean():.1f} ({df['TRTDUR'].std():.1f})"],
    'Median':[df['TRTDUR'].median()],
    'Min, Max':[f"{int(df['TRTDUR'].min())}, {int(df['TRTDUR'].max())}"]
}, index=['Total'])

exposure = pd.concat([exposure, total])
exposure

-   compliance

In [394]:
df = data['adsl']

df['DISCONT'] = df['DISCONFL'].apply(lambda x: 'Discontinued' if x == 'Y' else 'Completed')

ct = pd.crosstab(df['DISCONT'], df['ARM'])

pct = ct.div(ct.sum(axis=0), axis=1) * 100

disc_table = ct.astype(str) + " (" + pct.round(1).astype(str) + "%)"


total_ct = df['DISCONT'].value_counts()

total_pct = total_ct / len(df) * 100

disc_table['Total'] = total_ct.astype(str) + \
                      " (" + total_pct.round(1).astype(str) + "%)"

disc_table

## Efficacy Analysis

In [434]:
def mean_change_table_adqs_se(
    df,
    arm_col="TRTP",         
    param_col="PARAM",
    visit_col="AVISIT",
    aval_col="AVAL",
    base_col="BASE",
    flag_col="ANL01FL",      
    pop_flag_col="EFFFL",    
    use_flag=True,
    use_pop_flag=True,
    decimals=2,
):
    d = df.copy()

    # Analysis set filtering
    if use_pop_flag and pop_flag_col in d.columns:
        d = d[d[pop_flag_col] == "Y"]
    if use_flag and flag_col in d.columns:
        d = d[d[flag_col] == "Y"]

    # Numeric coercion
    d[aval_col] = pd.to_numeric(d[aval_col], errors="coerce")
    d[base_col] = pd.to_numeric(d[base_col], errors="coerce")

    # Change from baseline
    d["CHG_CALC"] = d[aval_col] - d[base_col]

    # Summary by ARM x PARAM x VISIT
    grp = d.groupby([arm_col, param_col, visit_col], dropna=False)["CHG_CALC"]
    summ = grp.agg(N="count", Mean="mean", SD="std").reset_index()
    summ["SE"] = summ["SD"] / np.sqrt(summ["N"])

    # Format Mean (SE)
    summ["Mean (SE)"] = (
        summ["Mean"].round(decimals).map(lambda x: f"{x:.{decimals}f}")
        + " ("
        + summ["SE"].round(decimals).fillna(0).map(lambda x: f"{x:.{decimals}f}")
        + ")"
    )

    # Pivot Mean(SE)
    wide = summ.pivot_table(
        index=[param_col, visit_col],
        columns=arm_col,
        values="Mean (SE)",
        aggfunc="first"
    )

    # Total column
    grp_tot = d.groupby([param_col, visit_col], dropna=False)["CHG_CALC"]
    tot = grp_tot.agg(N="count", Mean="mean", SD="std")
    tot["SE"] = tot["SD"] / np.sqrt(tot["N"])
    tot_fmt = (
        tot["Mean"].round(decimals).map(lambda x: f"{x:.{decimals}f}")
        + " ("
        + tot["SE"].round(decimals).fillna(0).map(lambda x: f"{x:.{decimals}f}")
        + ")"
    )
    wide["Total"] = tot_fmt

    # N table (optional but useful)
    n_wide = summ.pivot_table(
        index=[param_col, visit_col],
        columns=arm_col,
        values="N",
        aggfunc="first"
    )
    n_wide["Total"] = tot["N"]

    # Visit ordering by AVISITN if available
    if "AVISITN" in d.columns:
        visit_order = (
            d[[visit_col, "AVISITN"]]
            .drop_duplicates()
            .sort_values("AVISITN")
            .set_index(visit_col)["AVISITN"]
            .to_dict()
        )
        wide = wide.reset_index()
        wide["__v"] = wide[visit_col].map(visit_order)
        wide = wide.sort_values([param_col, "__v"]).drop(columns="__v").set_index([param_col, visit_col])

        n_wide = n_wide.reset_index()
        n_wide["__v"] = n_wide[visit_col].map(visit_order)
        n_wide = n_wide.sort_values([param_col, "__v"]).drop(columns="__v").set_index([param_col, visit_col])

    return wide, n_wide, d


# ===== usage =====
adas = data["adqsadas"]

mean_se_table, n_table, adas_used = mean_change_table_adqs_se(
    adas,
    arm_col="TRTP",  
    decimals=2
)

### N table

In [435]:
display(n_table)        # N

### Mean change from baseline

In [436]:
display(mean_se_table)  # Mean (SE)

### ANCOVA(Change from baseline ~ Treatment + Baseline)

In [445]:
df = data['adqsadas'].copy()

# analysis population
df = df[(df['EFFFL']=='Y') & (df['ANL01FL']=='Y')]

visits = ['Week 8','Week 16','Week 24']

rows = []

for v in visits:

    d = df[df['AVISIT']==v]

    model = smf.ols("CHG ~ TRTP + BASE", data=d).fit()

    params = model.params
    conf = model.conf_int()
    pvals = model.pvalues

    for trt in params.index:

        if trt.startswith("TRTP"):

            rows.append({
                "Visit": v,
                "Treatment": trt.split("T.")[-1].replace("]",""),
                "LS Mean Difference": round(params[trt],2),
                "95% CI": f"({conf.loc[trt,0]:.2f}, {conf.loc[trt,1]:.2f})",
                "p-value": round(pvals[trt],3)
            })

ancova_table = pd.DataFrame(rows)

ancova_table

> Across all visits, neither high-dose nor low-dose Xanomeline showed a
> statistically significant difference compared with placebo.

### MMRM, Mixed Model for Repeated Measures(CHG ~ TRTP \* AVISIT + BASE)

In [455]:
df = data['adqsadas'].copy()

df = df[(df['EFFFL']=='Y') & (df['ANL01FL']=='Y')]

df = df[df['AVISIT'] != 'Baseline']

mm = df[['USUBJID','TRTP','AVISIT','BASE','CHG']].dropna().reset_index(drop=True)

model = smf.mixedlm(
    "CHG ~ TRTP * AVISIT + BASE",
    data=mm,
    groups=mm["USUBJID"]
)

res = model.fit()

print(res.summary())

                          Mixed Linear Model Regression Results
Model:                       MixedLM            Dependent Variable:            CHG        
No. Observations:            8198               Method:                        REML       
No. Groups:                  234                Scale:                         299.2140   
Min. group size:             15                 Log-Likelihood:                -35048.3165
Max. group size:             45                 Converged:                     Yes        
Mean group size:             35.0                                                         
------------------------------------------------------------------------------------------
                                               Coef.  Std.Err.    z    P>|z| [0.025 0.975]
------------------------------------------------------------------------------------------
Intercept                                       2.980    0.606   4.915 0.000  1.792  4.169
TRTP[T.Xanomeline High Dos

In [464]:
def _fmt(x, d=2):
    if pd.isna(x):
        return ""
    return f"{x:.{d}f}"

def mmrm_lsmeans_and_diffs_fixedonly(
    res,
    mm,
    arm_col="TRTP",
    visit_col="AVISIT",
    base_col="BASE",
    visits=None,
    treatments=None,
    placebo_label=None,
    decimals=2
):
    # levels
    if visits is None:
        visits = list(pd.Series(mm[visit_col].unique()).dropna())
    if treatments is None:
        treatments = list(pd.Series(mm[arm_col].unique()).dropna())

    # keep category order if present
    if pd.api.types.is_categorical_dtype(mm[visit_col]):
        visits = [v for v in mm[visit_col].cat.categories if v in visits]
    else:
        visits = sorted(visits)

    if pd.api.types.is_categorical_dtype(mm[arm_col]):
        treatments = [t for t in mm[arm_col].cat.categories if t in treatments]

    # placebo default: first level if not given
    # if placebo_label is None:
    #     placebo_label = treatments[0]

    # if placebo_label not in treatments:
    #     raise ValueError(f"placebo_label='{placebo_label}' not found in {arm_col} levels: {treatments}")

    # prediction grid at mean(BASE)
    base_mean = pd.to_numeric(mm[base_col], errors="coerce").mean()

    grid = pd.DataFrame([(t, v) for v in visits for t in treatments], columns=[arm_col, visit_col])
    grid[base_col] = base_mean

    # align categories to training data (important)
    if pd.api.types.is_categorical_dtype(mm[arm_col]):
        grid[arm_col] = pd.Categorical(grid[arm_col], categories=mm[arm_col].cat.categories)
    if pd.api.types.is_categorical_dtype(mm[visit_col]):
        grid[visit_col] = pd.Categorical(grid[visit_col], categories=mm[visit_col].cat.categories)

    # build fixed-effects design matrix using model's design_info
    design_info = res.model.data.design_info
    X = patsy.build_design_matrices([design_info], grid, return_type="dataframe")[0]

    # fixed effects only
    fe_names = list(res.fe_params.index)
    beta = res.fe_params.loc[fe_names].values

    cov_all = res.cov_params()

    # ⭐ subset covariance to fixed-effect names
    if isinstance(cov_all, pd.DataFrame):
        covb = cov_all.loc[fe_names, fe_names].values
    else:
        # fallback if ndarray; assume fixed effects come first
        covb = np.asarray(cov_all)[:len(fe_names), :len(fe_names)]

    # also align X to fe_names (in case X has extra cols ordering mismatch)
    X = X[fe_names].values

    # LSMeans and SE
    pred = X @ beta
    var_pred = np.einsum("ij,jk,ik->i", X, covb, X)
    se_pred = np.sqrt(np.maximum(var_pred, 0))

    grid_out = grid.copy()
    grid_out["LSMean"] = pred
    grid_out["SE"] = se_pred
    grid_out["LSMean (SE)"] = grid_out["LSMean"].map(lambda x: _fmt(x, decimals)) + \
                              " (" + grid_out["SE"].map(lambda x: _fmt(x, decimals)) + ")"

    lsmean_table = (
        grid_out.pivot(index=visit_col, columns=arm_col, values="LSMean (SE)")
        .loc[visits, treatments]
    )

    # Differences vs placebo (Wald z)
    diffs = []
    # To access each row's X, rebuild as DataFrame with fe_names columns
    X_df = pd.DataFrame(X, columns=fe_names)

    for v in visits:
        idx_v = grid_out[grid_out[visit_col] == v].index

        # placebo row index
        idx_p = grid_out[(grid_out[visit_col] == v) & (grid_out[arm_col] == placebo_label)].index[0]
        x_p = X_df.loc[idx_p].values

        for t in treatments:
            if t == placebo_label:
                continue
            idx_t = grid_out[(grid_out[visit_col] == v) & (grid_out[arm_col] == t)].index[0]
            x_t = X_df.loc[idx_t].values

            c = x_t - x_p
            diff = c @ beta
            se = np.sqrt(np.maximum(c @ covb @ c, 0))
            z = diff / se if se > 0 else np.nan
            p = 2 * (1 - stats.norm.cdf(abs(z))) if pd.notna(z) else np.nan
            ci_low = diff - 1.96 * se
            ci_high = diff + 1.96 * se

            diffs.append({
                "Visit": v,
                "Treatment": t,
                "LS Mean Diff vs Placebo": _fmt(diff, decimals),
                "SE": _fmt(se, decimals),
                "95% CI": f"({_fmt(ci_low, decimals)}, {_fmt(ci_high, decimals)})",
                "p-value": "<0.001" if (pd.notna(p) and p < 0.001) else (_fmt(p, 3) if pd.notna(p) else "")
            })

    diff_table = pd.DataFrame(diffs)
    diff_table["Visit"] = pd.Categorical(diff_table["Visit"], categories=visits, ordered=True)
    diff_table = diff_table.sort_values(["Visit", "Treatment"]).reset_index(drop=True)

    return lsmean_table, diff_table, base_mean


# ===== usage =====
lsmeans, diffs, base_used = mmrm_lsmeans_and_diffs_fixedonly(
    res=res,
    mm=mm,
    arm_col="TRTP",
    visit_col="AVISIT",
    base_col="BASE",
    visits=["Week 8","Week 16","Week 24"],
    placebo_label="Placebo",
    decimals=2
)

# print(f"BASE fixed at mean(BASE) = {base_used:.2f}\n")

In [467]:
display(lsmeans)

-   There is no constant pattern by visits.

In [468]:
display(diffs)

> Across all visits, neither high-dose nor low-dose Xanomeline showed a
> statistically significant difference compared with placebo.

## Primary endpoint

-   adas-cog change from baseline

In [469]:
data['adqsadas'].head()

In [ ]:
data['adqscibc'].head()

-   Safety Analysis
    -   TEAE
    -   serious AE
    -   AE leading to discontinuation

In [330]:
data['adae']

-   Laboratory Analysis
    -   mean chang from baseline
    -   shift tables

In [332]:
data['adlbc']

In [333]:
data['adlbh']

-   Vital Signs
    -   systolic BP
    -   diastolic BP
    -   heart rate
    -   weight

In [334]:
data['advs']

-   Time-to-event Analysis
    -   Kaplan-Meier curve
    -   median survival
    -   Cox model

In [335]:
data['adtte']

-   Demographics

-   Table 14.1

    -   Demographic and Baseline Characteristics

In [ ]:
Columns
Placebo
Low Dose
High Dose
Total

Rows

Age mean (SD)
Sex n (%)
Race n (%)
Baseline ADAS score
Weight
Disease duration

-   Disposition
-   Table 14.2
    -   Patient Disposition

Randomized Treated Completed Discontinued

In [ ]:
Exposure
Table 14.3

Treatment Exposure Summary

Treatment duration
Mean exposure
Dose interruptions

Primary efficacy Table 14.4

Change from Baseline in ADAS-Cog

Baseline Week 12 Week 26 Mean change Treatment difference 95% CI p-value

In [ ]:
Secondary efficacy
Table 14.5

CIBIC+ Score

Improved
No change
Worsened

In [ ]:
Safety tables
Table 14.6

Patients with ≥1 TEAE

Placebo
Low dose
High dose

In [ ]:
Table 14.7

TEAE by System Organ Class

Example

Cardiac disorders
GI disorders
Nervous system disorders
Psychiatric disorders

In [ ]:
Table 14.8

Top Preferred Terms

Headache
Nausea
Fatigue
Dizziness

In [ ]:
Laboratory
Table 14.9

Laboratory Parameters – Change from Baseline

ALT
AST
Creatinine
Hemoglobin

In [ ]:
Table 14.10

Laboratory Shift Table

Low → Normal
Normal → High
High → Normal

In [ ]:
Vital signs
Table 14.11

Vital Signs Summary

Systolic BP
Diastolic BP
Heart rate
Weight

In [ ]:
Figures
Figure 14.1

ADAS-Cog change from baseline over time

(line plot)

In [ ]:
Figure 14.2

Kaplan-Meier survival curve

(dataset: ADTTE)

In [ ]:
Figure 14.3

AE incidence bar chart

## ADSL

> The first is ADSL (Subject Level Analysis Dataset), a
> one-recordper-subject structure that contains subject-level
> attributes. Because of its structure, it can be merged onto any other
> clinical dataset, including other ADaM datasets and SDTM datasets.

In [6]:
data['adsl'].head()

In [7]:
metadata['adsl'].column_names_to_labels

# Table

## Table 11-1. Demographic Characteristics

In [289]:
data2 = data['adsl'].copy()

data2['RACE2'] = data2['RACE'].replace({
    'WHITE': 'White/Caucasian',
    'BLACK OR AFRICAN AMERICAN': 'Other',
    'AMERICAN INDIAN OR ALASKA NATIVE': 'Other'
})

In [290]:
n_counts = data['adsl'].groupby('ARM')['USUBJID'].nunique()
total_n = data['adsl']['USUBJID'].nunique()
n_counts['Total'] = total_n

In [316]:
def mean_min_max_table(df, var, group_col='ARM', total_label='Total',
                       mean_digits=1, minmax_digits=0):

    arm_summary = (
        df.groupby(group_col)[var]
          .agg(['mean', 'min', 'max'])
    )

    total_summary = (
        df[var]
          .agg(['mean', 'min', 'max'])
          .to_frame().T
    )
    total_summary.index = [total_label]

    final = pd.concat([arm_summary, total_summary])

    final[f'{var} Mean (Min–Max)'] = (
        final['mean'].round(mean_digits).astype(str)
        + " ("
        + final['min'].round(minmax_digits).astype(int).astype(str)
        + "–"
        + final['max'].round(minmax_digits).astype(int).astype(str)
        + ")"
    )

    return final[[f'{var} Mean (Min–Max)']]

In [ ]:
def categorical_table(df, var, group_col='ARM', total_label='Total', pct_digits=0):

    ct = pd.crosstab(df[var], df[group_col])

    pct = ct.div(ct.sum(axis=0), axis=1) * 100

    formatted = (
        ct.astype(str) + " (" +
        pct.round(pct_digits).astype(int).astype(str) + "%)"
    )

    total_n = df.shape[0]
    total_ct = df[var].value_counts()
    total_pct = total_ct / total_n * 100

    formatted[total_label] = (
        total_ct.astype(str) + " (" +
        total_pct.round(pct_digits).astype(int).astype(str) + "%)"
    )

    formatted.columns.name = None

    return formatted

In [313]:
Table11_1 = pd.concat([mean_min_max_table(data['adsl'],'AGE').T,
                    categorical_table(data['adsl'],'SEX').reset_index().set_index('SEX').reindex(['M', 'F']),
                    categorical_table(data2,'RACE2').reset_index().set_index('RACE2').reindex(['White/Caucasian', 'Other']),
                    mean_min_max_table(data['adsl'],'EDUCLVL').T,
                   ])

In [314]:
Table11_1.columns = [
    f"{col} (n={n_counts[col]})" if col in n_counts.index
    else col
    for col in final2.columns
]

In [315]:
Table11_1

In [319]:
data['adtte']